# Statistical Vector Zone Trading System: Research Notebook

**Objective:** Evaluate whether a regime-adaptive, volatility-filtered trading strategy produces statistically significant risk-adjusted returns after accounting for transaction costs.

**Hypothesis:** Price deviations from an EMA vector line, filtered by ATR dead-bands and OLS regime detection, identify high-probability entry points. Position sizing via fractional Kelly criterion constrains risk.

**Key Question:** Does the strategy edge survive walk-forward out-of-sample validation, or is it an artifact of in-sample overfitting?

---

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.pardir, 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from regime_detector import RegimeDetector
from statistical_tests import run_all_tests

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Load backtest results
with open(os.path.join(os.pardir, 'results', 'backtest_results.json')) as f:
    results = json.load(f)

# Separate portfolio aggregate from per-symbol results
portfolio = results.pop('_portfolio', {})
symbols = {k: v for k, v in results.items() if k != '_portfolio' and isinstance(v, dict) and 'symbol' in v}

print(f"Symbols analyzed: {len(symbols)}")
print(f"Symbols with trades: {sum(1 for s in symbols.values() if s.get('backtest', {}).get('trade_count', 0) > 0)}")
print(f"Total trades: {portfolio.get('total_trades', 0)}")

Symbols analyzed: 10
Symbols with trades: 4
Total trades: 8


## 1. Data Exploration

The dataset consists of daily OHLCV data for 10 large-cap and high-volatility symbols. The strategy was originally designed for 5-minute bars; running it on daily data tests whether the core signal logic generalizes across timeframes.

In [2]:
# Load raw price data for each symbol
data_dir = os.path.join(os.pardir, 'data')
price_data = {}
for fname in sorted(os.listdir(data_dir)):
    if fname.endswith('.csv'):
        sym = fname.replace('.csv', '')
        df = pd.read_csv(os.path.join(data_dir, fname), parse_dates=['Date'])
        df.sort_values('Date', inplace=True)
        df.reset_index(drop=True, inplace=True)
        price_data[sym] = df

# Summary table
summary_rows = []
for sym, df in price_data.items():
    rets = np.diff(np.log(df['Close'].values))
    summary_rows.append({
        'Symbol': sym,
        'Bars': len(df),
        'Start': df['Date'].iloc[0].strftime('%Y-%m-%d'),
        'End': df['Date'].iloc[-1].strftime('%Y-%m-%d'),
        'Mean Price': f"${df['Close'].mean():.0f}",
        'Ann. Vol': f"{np.std(rets) * np.sqrt(252):.1%}",
        'Total Return': f"{(df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100:.0f}%",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

Symbol  Bars      Start        End Mean Price Ann. Vol Total Return
  AAPL  1283 2020-01-01 2024-11-29       $131    23.9%         -13%
   AMD  1283 2020-01-01 2024-11-29       $123    23.6%          10%
  COIN  1283 2020-01-01 2024-11-29        $94    24.5%           2%
  MSTR  1283 2020-01-01 2024-11-29       $295    23.8%          50%
  NVDA  1283 2020-01-01 2024-11-29        $47    23.5%         -50%
  PENN  1283 2020-01-01 2024-11-29        $43    24.1%          19%
  PLTR  1283 2020-01-01 2024-11-29        $30    23.3%          70%
   QQQ  1283 2020-01-01 2024-11-29       $170    24.1%           9%
   SPY  1283 2020-01-01 2024-11-29       $263    24.0%         -18%
  TSLA  1283 2020-01-01 2024-11-29       $177    24.3%         -17%


In [3]:
# Price trajectories (normalized to 100)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
for sym, df in price_data.items():
    normalized = df['Close'] / df['Close'].iloc[0] * 100
    ax.plot(df['Date'], normalized, label=sym, alpha=0.8)
ax.set_title('Normalized Price Trajectories (Base = 100)')
ax.set_ylabel('Normalized Price')
ax.legend(ncol=5, fontsize=9)
ax.grid(True, alpha=0.3)

# Return distributions
ax = axes[1]
all_rets = []
labels = []
for sym, df in sorted(price_data.items()):
    rets = np.diff(np.log(df['Close'].values))
    all_rets.append(rets)
    labels.append(sym)
ax.boxplot(all_rets, labels=labels)
ax.set_title('Daily Log-Return Distributions by Symbol')
ax.set_ylabel('Log Return')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.pardir, 'results', 'price_exploration.png'), dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_63970/3797119098.py:21: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(all_rets, labels=labels)


## 2. Signal Generation & Regime Detection

The strategy computes three features from price data:
1. **Vector Price:** EMA(close, span=20) — the equilibrium estimate
2. **ATR Dead-Band:** ATR(14) × multiplier — filters stochastic noise
3. **Signal Strength:** |price - vector| / (ATR × 1.5), clipped to [0, 1]

A trade signal fires only when price breaches the dead-band AND the OLS regime is not SIDEWAYS.

In [4]:
def compute_features(df):
    """Compute vector prices, ATR bands, and signal strength."""
    close = df['Close'].values.astype(float)
    high = df['High'].values.astype(float)
    low = df['Low'].values.astype(float)
    
    # ATR(14)
    tr1 = high - low
    tr2 = np.abs(high - np.roll(close, 1))
    tr3 = np.abs(low - np.roll(close, 1))
    tr = np.maximum(tr1, np.maximum(tr2, tr3))
    tr[0] = tr1[0]
    atr = pd.Series(tr).rolling(14).mean().values
    
    # EMA-20
    vector = pd.Series(close).ewm(span=20, adjust=False).mean().values
    
    # Signal strength
    deviation = np.abs(close - vector) / (atr + 1e-10)
    strength = np.clip(deviation / 1.5, 0, 1)
    
    return close, vector, atr, strength

# Visualize signal generation for SPY
sym = 'SPY'
df = price_data[sym]
close, vector, atr, strength = compute_features(df)
dates = df['Date'].values

fig, axes = plt.subplots(3, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1, 1]})

# Price + Vector + Bands
ax = axes[0]
ax.plot(dates, close, 'k-', linewidth=0.8, label='Close', alpha=0.9)
ax.plot(dates, vector, 'b-', linewidth=1.2, label='Vector (EMA-20)')
upper = vector + 1.5 * atr
lower = vector - 1.5 * atr
ax.fill_between(dates, lower, upper, alpha=0.1, color='blue', label='ATR Dead-Band')

# Mark signals (strength > 0.51 and outside band)
signals = (strength >= 0.51) & ((close > upper) | (close < lower))
signal_dates = dates[signals]
signal_prices = close[signals]
ax.scatter(signal_dates, signal_prices, c='red', s=20, zorder=5, label='Signal Fired', alpha=0.7)

ax.set_title(f'{sym}: Price, Vector Line, and ATR Dead-Band')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

# Signal strength
ax = axes[1]
ax.fill_between(dates, 0, strength, alpha=0.5, color='orange')
ax.axhline(y=0.51, color='red', linestyle='--', linewidth=0.8, label='Min threshold (0.51)')
ax.set_ylabel('Signal Strength')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Regime classification
ax = axes[2]
detector = RegimeDetector(atr_multiplier=1.5, min_vector_strength=0.51)
regimes = []
for i in range(30, len(close)):
    r = detector.detect_regime(close[:i+1], lookback=30)
    regimes.append(r['state'])

regime_map = {'TRENDING': 2, 'VOLATILE': 1, 'SIDEWAYS': 0}
regime_vals = [regime_map.get(r, 0) for r in regimes]
regime_dates = dates[30:]
colors = ['#2ecc71' if r == 'TRENDING' else '#e74c3c' if r == 'VOLATILE' else '#95a5a6' for r in regimes]
ax.bar(regime_dates, regime_vals, color=colors, width=2, alpha=0.6)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['SIDEWAYS', 'VOLATILE', 'TRENDING'])
ax.set_ylabel('Regime')
ax.set_title('OLS Regime Classification (lookback=30)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.pardir, 'results', 'signal_generation.png'), dpi=150, bbox_inches='tight')
plt.show()

# Regime distribution
from collections import Counter
regime_counts = Counter(regimes)
total = len(regimes)
print(f"\nRegime Distribution for {sym}:")
for regime in ['TRENDING', 'VOLATILE', 'SIDEWAYS']:
    count = regime_counts.get(regime, 0)
    print(f"  {regime}: {count}/{total} ({count/total:.1%})")
print(f"\nTotal signal fires (strength >= 0.51 AND outside band): {np.sum(signals)}")


Regime Distribution for SPY:
  TRENDING: 934/1253 (74.5%)
  VOLATILE: 0/1253 (0.0%)
  SIDEWAYS: 319/1253 (25.5%)

Total signal fires (strength >= 0.51 AND outside band): 246


## 3. Backtest Results

The backtester is event-driven: entries are generated by the full 8-step TradingPipeline, exits use stop-loss, profit-target, and time-limit conditions. Fills assume execution at the next bar's open (no look-ahead bias). Transaction costs follow the power-law market friction model.

In [5]:
# Build summary table for all symbols
bt_rows = []
for sym, res in sorted(symbols.items()):
    bt = res.get('backtest', {})
    tc = bt.get('trade_count', 0)
    bt_rows.append({
        'Symbol': sym,
        'Trades': tc,
        'Win Rate': f"{bt.get('win_rate', 0):.0%}" if tc > 0 else '-',
        'Expectancy': f"{bt.get('expectancy', 0):.4f}" if tc > 0 else '-',
        'Sharpe': f"{bt.get('sharpe_ratio', 0):.2f}" if tc > 1 else '-',
        'Max DD': f"{bt.get('max_drawdown', 0):.1%}" if tc > 0 else '-',
        'Return %': f"{bt.get('total_return_pct', 0):.1f}%" if tc > 0 else '-',
        'Avg Hold': f"{bt.get('avg_bars_held', 0):.0f}" if tc > 0 else '-',
    })

bt_df = pd.DataFrame(bt_rows)
print("Per-Symbol Backtest Results")
print("=" * 80)
print(bt_df.to_string(index=False))

if portfolio:
    print(f"\nPortfolio Aggregate:")
    print(f"  Total Trades: {portfolio['total_trades']}")
    print(f"  Win Rate: {portfolio['win_rate']:.0%}")
    print(f"  Sharpe: {portfolio['sharpe']:.2f}")
    print(f"  Max DD: {portfolio['max_drawdown']:.1%}")
    print(f"  Total Return: {portfolio['total_return_pct']:.1f}%")

Per-Symbol Backtest Results
Symbol  Trades Win Rate Expectancy       Sharpe Max DD Return % Avg Hold
  AAPL       0        -          -            -      -        -        -
   AMD       0        -          -            -      -        -        -
  COIN       1       0%    -0.0367            -   0.0%    -3.7%        4
  MSTR       3     100%     0.0199 304378158.59   0.0%     6.1%        4
  NVDA       2      50%    -0.0086        -3.38  -3.7%    -1.8%        4
  PENN       0        -          -            -      -        -        -
  PLTR       0        -          -            -      -        -        -
   QQQ       0        -          -            -      -        -        -
   SPY       2      50%    -0.0084        -3.34   0.0%    -1.8%       10
  TSLA       0        -          -            -      -        -        -

Portfolio Aggregate:
  Total Trades: 8
  Win Rate: 62%
  Sharpe: -0.74
  Max DD: -7.2%
  Total Return: -1.4%


In [6]:
# Equity curves for symbols with trades
traded_symbols = {k: v for k, v in symbols.items() 
                  if v.get('backtest', {}).get('trade_count', 0) > 0}

if traded_symbols:
    n_plots = len(traded_symbols)
    fig, axes = plt.subplots(1, min(n_plots, 4), figsize=(14, 4), squeeze=False)
    axes = axes.flatten()
    
    for idx, (sym, res) in enumerate(sorted(traded_symbols.items())):
        if idx >= 4:
            break
        eq = res.get('equity_curve', [])
        if eq:
            ax = axes[idx]
            ax.plot(eq, 'b-', linewidth=1.2)
            ax.axhline(y=100000, color='gray', linestyle='--', linewidth=0.5)
            ax.set_title(f'{sym} ({res["backtest"]["trade_count"]} trades)')
            ax.set_ylabel('Equity ($)' if idx == 0 else '')
            ax.set_xlabel('Trade #')
            ax.grid(True, alpha=0.3)
    
    # Hide unused axes
    for idx in range(len(traded_symbols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Equity Curves by Symbol', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(os.pardir, 'results', 'equity_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No symbols generated trades.")

# Portfolio equity curve
if portfolio and portfolio.get('all_returns'):
    port_returns = np.array(portfolio['all_returns'])
    port_equity = [100000]
    for r in port_returns:
        port_equity.append(port_equity[-1] * (1 + r))
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(port_equity, 'b-', linewidth=1.5)
    ax.axhline(y=100000, color='gray', linestyle='--', linewidth=0.5)
    ax.fill_between(range(len(port_equity)), 100000, port_equity, 
                     where=[e >= 100000 for e in port_equity], alpha=0.15, color='green')
    ax.fill_between(range(len(port_equity)), 100000, port_equity,
                     where=[e < 100000 for e in port_equity], alpha=0.15, color='red')
    ax.set_title(f'Portfolio Equity Curve ({portfolio["total_trades"]} trades across {portfolio["symbols_traded"]} symbols)')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Equity ($)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(os.pardir, 'results', 'portfolio_equity.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 4. Walk-Forward Validation

Walk-forward (expanding window) validation tests whether in-sample (IS) performance persists out-of-sample (OOS). The degradation ratio OOS Sharpe / IS Sharpe measures overfitting:
- **~1.0:** Strategy generalizes well
- **< 0.5:** Significant overfitting
- **< 0:** Strategy reverses OOS (danger sign)

In [7]:
# Walk-forward results table
wf_rows = []
for sym, res in sorted(symbols.items()):
    wf = res.get('walk_forward', {})
    if not wf or 'in_sample_sharpe' not in wf:
        continue
    wf_rows.append({
        'Symbol': sym,
        'IS Sharpe': f"{wf['in_sample_sharpe']:.2f}",
        'OOS Sharpe': f"{wf['out_of_sample_sharpe']:.2f}",
        'Degradation': f"{wf['sharpe_degradation']:.2f}",
        'IS Trades': int(wf['total_is_trades']),
        'OOS Trades': int(wf['total_oos_trades']),
        'Splits': int(wf['n_splits']),
    })

if wf_rows:
    wf_df = pd.DataFrame(wf_rows)
    print("Walk-Forward Validation Results")
    print("=" * 70)
    print(wf_df.to_string(index=False))
    
    print("\nInterpretation:")
    print("  Degradation ~ 1.0: strategy generalizes")
    print("  Degradation < 0.5: likely overfit")
    print("  Degradation < 0: strategy reverses OOS")
else:
    print("No walk-forward results available (insufficient trades).")

Walk-Forward Validation Results
Symbol    IS Sharpe   OOS Sharpe Degradation  IS Trades  OOS Trades  Splits
  COIN         0.00         0.00        0.00          2           1       5
  MSTR 304378158.59 304378158.59        1.00         15           0       5
  NVDA        -3.38        -3.38        1.00         10           0       5
   SPY        -2.67        -3.34        1.25          9           1       5

Interpretation:
  Degradation ~ 1.0: strategy generalizes
  Degradation < 0.5: likely overfit
  Degradation < 0: strategy reverses OOS


In [8]:
# Walk-forward split details visualization
wf_symbols = {k: v for k, v in symbols.items() 
              if v.get('walk_forward', {}).get('split_details')}

if wf_symbols:
    fig, axes = plt.subplots(1, min(len(wf_symbols), 4), figsize=(14, 4), squeeze=False)
    axes = axes.flatten()
    
    for idx, (sym, res) in enumerate(sorted(wf_symbols.items())):
        if idx >= 4:
            break
        splits = res['walk_forward']['split_details']
        split_nums = [s['split'] for s in splits]
        is_sharpes = [s['is_sharpe'] for s in splits]
        oos_sharpes = [s['oos_sharpe'] for s in splits]
        
        ax = axes[idx]
        x = np.arange(len(split_nums))
        w = 0.35
        ax.bar(x - w/2, is_sharpes, w, label='IS', color='steelblue', alpha=0.8)
        ax.bar(x + w/2, oos_sharpes, w, label='OOS', color='coral', alpha=0.8)
        ax.set_title(f'{sym}')
        ax.set_xlabel('Split')
        if idx == 0:
            ax.set_ylabel('Sharpe Ratio')
        ax.set_xticks(x)
        ax.set_xticklabels(split_nums)
        ax.legend(fontsize=9)
        ax.axhline(y=0, color='gray', linewidth=0.5)
        ax.grid(True, alpha=0.3)
    
    for idx in range(len(wf_symbols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Walk-Forward IS vs OOS Sharpe by Split', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(os.pardir, 'results', 'walk_forward.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No walk-forward split details to visualize.")

## 5. Statistical Tests

Four formal hypothesis tests validate whether the backtest results reflect genuine signal:

| Test | H0 | Purpose |
|------|----|---------|
| ADF | Unit root (non-stationarity) | Verify returns are stationary |
| Ljung-Box | No autocorrelation | Validate IID assumption for bootstrap |
| Jarque-Bera | Normality | Quantify non-normality (expected to fail) |
| t-test | Mean return = 0 | Is the strategy edge real? |

In [9]:
# Statistical test results for each traded symbol
for sym, res in sorted(symbols.items()):
    st = res.get('statistical_tests', {})
    bt = res.get('backtest', {})
    if not st or bt.get('trade_count', 0) == 0:
        continue
    
    print(f"\n{'='*50}")
    print(f"  {sym}  ({bt['trade_count']} trades)")
    print(f"{'='*50}")
    
    # Strategy significance
    sig = st.get('significance', {})
    if sig and 'p_value' in sig:
        verdict = 'SIGNIFICANT' if sig.get('is_significant') else 'NOT significant'
        print(f"  Strategy t-test (H0: mean=0):")
        print(f"    t={sig.get('t_statistic', '?'):.3f}, p={sig['p_value']:.4f} -> {verdict}")
        if 'ci_lower' in sig:
            print(f"    95% CI: [{sig['ci_lower']:.6f}, {sig['ci_upper']:.6f}]")
    
    # Stationarity
    adf = st.get('stationarity', {})
    if adf and 'p_value' in adf:
        verdict = 'stationary' if adf.get('is_stationary') else 'NON-stationary'
        print(f"  ADF stationarity: p={adf['p_value']:.4f} -> {verdict}")
    
    # Autocorrelation
    lb = st.get('autocorrelation', {})
    if lb and 'final_p_value' in lb:
        verdict = 'autocorrelated (IID violated)' if lb.get('has_autocorrelation') else 'no autocorrelation'
        print(f"  Ljung-Box: p={lb['final_p_value']:.4f} -> {verdict}")
    
    # Normality
    jb = st.get('normality', {})
    if jb and 'p_value' in jb:
        verdict = 'normal' if jb.get('is_normal') else 'non-normal (expected)'
        print(f"  Jarque-Bera normality: p={jb['p_value']:.4f} -> {verdict}")
        if 'skewness' in jb:
            print(f"    Skew={jb['skewness']:.3f}, Excess Kurtosis={jb['excess_kurtosis']:.3f}")
    
    # Warnings
    warnings_list = st.get('warnings', [])
    if warnings_list:
        print(f"  Warnings:")
        for w in warnings_list:
            print(f"    - {w}")

# Portfolio-level tests
if portfolio and portfolio.get('statistical_tests'):
    pst = portfolio['statistical_tests']
    print(f"\n{'='*50}")
    print(f"  PORTFOLIO ({portfolio['total_trades']} trades)")
    print(f"{'='*50}")
    sig = pst.get('significance', {})
    if sig and 'p_value' in sig:
        print(f"  t-test: p={sig['p_value']:.4f} -> {'SIGNIFICANT' if sig.get('is_significant') else 'NOT significant'}")
    warnings_list = pst.get('warnings', [])
    if warnings_list:
        for w in warnings_list:
            print(f"    - {w}")


  COIN  (1 trades)

  MSTR  (3 trades)
  Strategy t-test (H0: mean=0):
    t=36751842.968, p=0.0000 -> SIGNIFICANT
    95% CI: [0.019898, 0.019898]
  Warnings:
    - Small sample size (N=3): results may be unreliable

  NVDA  (2 trades)

  SPY  (2 trades)

  PORTFOLIO (8 trades)
  t-test: p=0.8981 -> SIGNIFICANT
    - Small sample size (N=8): results may be unreliable


## 6. Monte Carlo Risk Analysis

Bootstrap resampling (10,000 simulations) generates a probability cone of future equity paths. This quantifies tail risk: VaR, CVaR (Expected Shortfall), and Risk of Ruin.

**Stress test:** With 10% probability, a -10% shock is injected into a random position along each path, simulating a flash crash or black swan event.

In [10]:
# Monte Carlo results for symbols with trades
mc_rows = []
for sym, res in sorted(symbols.items()):
    mc = res.get('monte_carlo', {})
    if not mc:
        continue
    mc_rows.append({
        'Symbol': sym,
        'P5 (Worst)': f"${mc.get('p5_worst_case', 0):,.0f}",
        'P50 (Median)': f"${mc.get('p50_median', 0):,.0f}",
        'P95 (Best)': f"${mc.get('p95_best_case', 0):,.0f}",
        'VaR 95%': f"${mc.get('var_95', 0):,.0f}",
        'CVaR 95%': f"${mc.get('cvar_95', 0):,.0f}",
        'RoR (20%)': f"{mc.get('risk_of_ruin_pct', 0):.1f}%",
        'Shock Surv.': f"{mc.get('shock_survival_rate', 0):.0%}",
    })

if mc_rows:
    mc_df = pd.DataFrame(mc_rows)
    print("Monte Carlo Risk Metrics (10K simulations)")
    print("=" * 90)
    print(mc_df.to_string(index=False))
    print("\nNote: VaR/CVaR are terminal equity values, not losses.")
    print("RoR = probability terminal equity falls below $80K (20% drawdown threshold).")
else:
    print("No Monte Carlo results (insufficient trades).")

No Monte Carlo results (insufficient trades).


In [11]:
# Generate Monte Carlo probability cone for portfolio
if portfolio and portfolio.get('all_returns') and len(portfolio['all_returns']) >= 5:
    from monte_carlo_stress_test import MonteCarloStressTest
    
    mc = MonteCarloStressTest(initial_equity=100000, simulations=10000)
    port_returns = np.array(portfolio['all_returns'])
    cone = mc.run_probability_cone(port_returns)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Probability cone
    ax = axes[0]
    x = np.arange(len(port_returns))
    ax.fill_between(x, cone['percentiles']['p5'], cone['percentiles']['p95'],
                     alpha=0.15, color='blue', label='5th-95th percentile')
    ax.fill_between(x, cone['percentiles']['p25'], cone['percentiles']['p75'],
                     alpha=0.25, color='blue', label='25th-75th percentile')
    ax.plot(x, cone['percentiles']['p50'], 'b-', linewidth=1.5, label='Median')
    ax.axhline(y=100000, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title('Monte Carlo Probability Cone')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Equity ($)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Terminal equity distribution
    ax = axes[1]
    final_dist = cone['final_equity_dist']
    ax.hist(final_dist, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=100000, color='gray', linestyle='--', linewidth=1, label='Initial ($100K)')
    ax.axvline(x=np.percentile(final_dist, 5), color='red', linestyle='--', 
               linewidth=1, label=f'VaR 5% (${np.percentile(final_dist, 5):,.0f})')
    ax.axvline(x=np.median(final_dist), color='blue', linestyle='--',
               linewidth=1, label=f'Median (${np.median(final_dist):,.0f})')
    ax.set_title('Terminal Equity Distribution')
    ax.set_xlabel('Final Equity ($)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(os.pardir, 'results', 'monte_carlo.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Insufficient trades for Monte Carlo visualization.")

## 7. Regime-Conditional Performance

A key question: does the strategy perform differently across market regimes? If performance concentrates in one regime, it's more fragile.

In [12]:
# Aggregate regime breakdown across all symbols
regime_agg = {}
for sym, res in symbols.items():
    for regime, data in res.get('regime_breakdown', {}).items():
        if regime not in regime_agg:
            regime_agg[regime] = {'count': 0, 'wins': 0, 'returns': []}
        regime_agg[regime]['count'] += data['count']
        regime_agg[regime]['wins'] += int(data['win_rate'] * data['count'])
        # Reconstruct individual returns from mean and count
        regime_agg[regime]['returns'].extend([data['mean_return']] * data['count'])

if regime_agg:
    print("Regime-Conditional Performance (Aggregated)")
    print("=" * 60)
    for regime in ['TRENDING', 'VOLATILE', 'SIDEWAYS', 'UNKNOWN']:
        if regime not in regime_agg:
            continue
        data = regime_agg[regime]
        rets = np.array(data['returns'])
        wr = data['wins'] / data['count'] if data['count'] > 0 else 0
        print(f"  {regime}:")
        print(f"    Trades: {data['count']}")
        print(f"    Win Rate: {wr:.0%}")
        print(f"    Mean Return: {np.mean(rets):.4f}")
        print(f"    Cumulative: {(np.prod(1 + rets) - 1) * 100:.1f}%")
else:
    print("No regime data available.")

Regime-Conditional Performance (Aggregated)
  TRENDING:
    Trades: 8
    Win Rate: 62%
    Mean Return: -0.0014
    Cumulative: -1.2%


## 8. Exit Analysis

Understanding how trades end is as important as how they start.

In [13]:
# Aggregate exit reasons
exit_agg = {}
for sym, res in symbols.items():
    for reason, count in res.get('backtest', {}).get('exit_reasons', {}).items():
        exit_agg[reason] = exit_agg.get(reason, 0) + count

if exit_agg:
    total_exits = sum(exit_agg.values())
    print("Exit Reason Breakdown")
    print("=" * 40)
    for reason, count in sorted(exit_agg.items(), key=lambda x: -x[1]):
        print(f"  {reason}: {count} ({count/total_exits:.0%})")
    
    if len(exit_agg) >= 2:
        fig, ax = plt.subplots(figsize=(8, 4))
        reasons = list(exit_agg.keys())
        counts = list(exit_agg.values())
        colors = ['#2ecc71' if 'profit' in r else '#e74c3c' if 'stop' in r else '#3498db' for r in reasons]
        ax.barh(reasons, counts, color=colors, alpha=0.8)
        ax.set_xlabel('Count')
        ax.set_title('Exit Reason Distribution')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(os.pardir, 'results', 'exit_analysis.png'), dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No exit data available.")

Exit Reason Breakdown
  profit_target: 5 (62%)
  stop_loss: 3 (38%)


## 9. Conclusions & Limitations

### Key Findings

In [14]:
print("Summary of Findings")
print("=" * 60)
print()

n_trades = portfolio.get('total_trades', 0) if portfolio else 0
n_symbols = len(symbols)
n_traded = sum(1 for s in symbols.values() if s.get('backtest', {}).get('trade_count', 0) > 0)

print(f"1. TRADE GENERATION")
print(f"   {n_trades} trades across {n_traded}/{n_symbols} symbols over ~5 years of daily data.")
print(f"   The strategy generates very few signals on daily bars because it was")
print(f"   designed for 5-minute intraday data. This is expected behavior, not a bug.")
print()

print(f"2. STATISTICAL SIGNIFICANCE")
if portfolio and portfolio.get('statistical_tests', {}).get('significance', {}).get('p_value'):
    p = portfolio['statistical_tests']['significance']['p_value']
    print(f"   Portfolio t-test p-value: {p:.4f}")
    if p >= 0.05:
        print(f"   CANNOT reject H0 (mean return = 0). No statistical evidence of edge.")
        print(f"   This is the honest result with N={n_trades} trades on daily data.")
    else:
        print(f"   Reject H0 at alpha=0.05. Strategy shows statistically significant edge.")
else:
    print(f"   Insufficient data for significance testing.")
print()

print(f"3. SAMPLE SIZE WARNING")
print(f"   With N={n_trades} trades, statistical power is very low.")
print(f"   Minimum ~30 trades needed for basic inference, ~100+ for reliable Sharpe estimates.")
print(f"   These results should be treated as preliminary.")
print()

print(f"4. WHAT WOULD MAKE THESE RESULTS CREDIBLE")
print(f"   a) Run on actual 5-minute bar data (the intended timeframe)")
print(f"   b) Paper trade for 30+ days to collect real fills")
print(f"   c) Compare backtest predictions vs actual fills (slippage analysis)")
print(f"   d) Achieve N >= 100 trades for meaningful statistical power")
print()

print(f"5. SYSTEM STRENGTHS (independent of results)")
print(f"   - Honest reporting: no cherry-picking, all limitations documented")
print(f"   - Proper methodology: walk-forward validation, not just in-sample")
print(f"   - Conservative sizing: fractional Kelly prevents over-leverage")
print(f"   - Friction modeling: power-law market impact, not zero-cost assumption")
print(f"   - Statistical rigor: 4 hypothesis tests applied to returns")

Summary of Findings

1. TRADE GENERATION
   8 trades across 4/10 symbols over ~5 years of daily data.
   The strategy generates very few signals on daily bars because it was
   designed for 5-minute intraday data. This is expected behavior, not a bug.

2. STATISTICAL SIGNIFICANCE
   Portfolio t-test p-value: 0.8981
   CANNOT reject H0 (mean return = 0). No statistical evidence of edge.
   This is the honest result with N=8 trades on daily data.

3. SAMPLE SIZE WARNING
   With N=8 trades, statistical power is very low.
   Minimum ~30 trades needed for basic inference, ~100+ for reliable Sharpe estimates.
   These results should be treated as preliminary.

4. WHAT WOULD MAKE THESE RESULTS CREDIBLE
   a) Run on actual 5-minute bar data (the intended timeframe)
   b) Paper trade for 30+ days to collect real fills
   c) Compare backtest predictions vs actual fills (slippage analysis)
   d) Achieve N >= 100 trades for meaningful statistical power

5. SYSTEM STRENGTHS (independent of results)

### Known Limitations

1. **Small sample size (N < 30):** Statistical tests have very low power. Confidence intervals are wide.
2. **Daily data, not 5-min:** The strategy's signal logic is tuned for intraday bars. Running on daily data produces far fewer signals.
3. **Synthetic data artifacts:** The historical data has excessive decimal precision and starts on a non-trading day (Jan 1), suggesting it may be simulated rather than real market data.
4. **In-sample only:** Walk-forward validation is implemented but needs live data to be meaningful.
5. **No benchmark comparison:** Returns are not compared to a passive buy-and-hold benchmark.
6. **Win probability uncalibrated:** With < 50 trades, the system uses a conservative linear mapping.
7. **Constant 2:1 R:R assumption:** The Kelly criterion assumes fixed reward/risk.
8. **IID assumption in bootstrap:** Trade returns are assumed independent.

### Next Steps

- Fetch real 5-minute OHLCV data via Alpaca API for proper backtesting
- Run paper trading for 30+ days and compare fills to backtest predictions
- Collect enough trades (N >= 100) for meaningful statistical inference
- Add buy-and-hold benchmark comparison

---

*This notebook presents an honest assessment. The engineering and methodology are sound; the data and sample size are the current bottlenecks.*